# Image Captioning — ResNet-101 + Transformer Decoder (MS COCO 2017)

## Architecture
| Component | Detail |
|---|---|
| **Visual Encoder** | ResNet-101 (ImageNet pre-trained) — spatial grid 7×7 → 49 patch tokens (2048-d) projected to `d_model=512` |
| **Transformer Decoder** | 6 layers: masked self-attn → cross-attn → FFN (Pre-LN for stability) |
| **Output head** | `Linear(512 → vocab_size~10K)` + softmax |

## Training
- **Loss**: cross-entropy with label smoothing=0.1 (padding ignored)
- **Teacher forcing**: ground-truth tokens fed as decoder input at every step
- **Data**: ~590K pairs from 118K COCO images × 5 captions each
- **Optimiser**: AdamW + cosine LR annealing + mixed precision (AMP on GPU)

In [ ]:
# Run once if packages are missing
%pip install torch torchvision tqdm matplotlib Pillow ipywidgets

^C
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
   ---------------------------------------- 0.0/114.5 MB ? eta -:--:--
   ---------------------------------------- 0.0/114.5 MB ? eta -:--:--
   ---------------------------------------- 0.0/114.5 MB 445.2 kB/s eta 0:04:18
   ---------------------------------------- 0.0/114.5 MB 393.8 kB/s eta 0:04:51
   ---------------------------------------- 0.1/114.5 MB 550.5 kB/s eta 0:03:28
   ---------------------------------------- 0.1/114.5 MB 438.9 kB/s eta 0:04:21
   ---------------------------------------- 0.1/114.5 MB 504.4 kB/s eta 0:03:47
   ---------------------------------------- 0.1/114.5 MB 481.4 kB/s eta 0:03:58
   ---------------------------------------- 0.1/114.5 MB 568.9 kB/s eta 0:03:22
   ---------------------------------------- 0.2/114.5 MB 583.1 kB/s eta 0:03:17
   ---------------------------------------- 0.2/114.5 MB 590.8 kB/s eta 0:03:14
   ---------------------------------------- 0.2/114.5 MB 599.0 kB/s eta 0:03

In [2]:
%pip install ipywidgets

Note: you may need to restart the kernel to use updated packages.


In [3]:
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")
print(f"CUDA version: {torch.version.cuda}")

CUDA available: True
GPU: NVIDIA GeForce RTX 4050 Laptop GPU
CUDA version: 12.1


In [4]:
import os
import json
import math
import random
import re
from collections import Counter
from pathlib import Path

from PIL import Image
import matplotlib.pyplot as plt
import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
import torchvision.models as models
import torchvision.transforms as transforms

from tqdm.notebook import tqdm

In [1]:
%pip install datasets huggingface-hub

  Using cached tqdm-4.67.3-py3-none-any.whl.metadata (57 kB)
  Using cached hf_xet-1.4.2-cp37-abi3-win_amd64.whl.metadata (4.9 kB)
  Using cached typer-0.24.1-py3-none-any.whl.metadata (16 kB)
  Using cached click-8.3.1-py3-none-any.whl.metadata (2.6 kB)
  Using cached shellingham-1.5.4-py2.py3-none-any.whl.metadata (3.5 kB)
  Using cached rich-14.3.3-py3-none-any.whl.metadata (18 kB)
  Using cached annotated_doc-0.0.4-py3-none-any.whl.metadata (6.6 kB)
  Using cached aiohappyeyeballs-2.6.1-py3-none-any.whl.metadata (5.9 kB)
  Using cached aiosignal-1.4.0-py3-none-any.whl.metadata (3.7 kB)
     ---------------------------------------- 0.0/82.2 kB ? eta -:--:--
     ---------------------------------------- 82.2/82.2 kB 4.5 MB/s eta 0:00:00
  Using cached markdown_it_py-4.0.0-py3-none-any.whl.metadata (7.3 kB)
  Using cached mdurl-0.1.2-py3-none-any.whl.metadata (1.6 kB)
   ---------------------------------------- 0.0/527.0 kB ? eta -:--:--
   ----------------------- ---------------- 307


[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
'''from datasets import Dataset, DatasetDict

# Authenticate with Hugging Face Hub (run once)
# If prompted, enter your HF token from https://huggingface.co/settings/tokens
from huggingface_hub import login
login()'''

## Configuration

In [ ]:
from pathlib import Path
from collections import Counter
import json
import re

In [ ]:
# In first code cell after imports
import os

# Disable CUDA graphs (can help with memory stability on laptops)
os.environ['CUDA_LAUNCH_BLOCKING'] = '1'

# Set optimal CUDA settings
torch.backends.cudnn.benchmark = True  # Auto-tune convolution kernels
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

In [ ]:
# -- Paths --------------------------------------------------------------------
'''DATA_DIR       = Path("data/coco")
TRAIN_ANN      = DATA_DIR / "annotations" / "captions_train2017.json"
VAL_ANN        = DATA_DIR / "annotations" / "captions_val2017.json"'''
# For HF Hub datasets, image paths are already full paths, so use current directory
TRAIN_IMG_DIR  = Path(".")
VAL_IMG_DIR    = Path(".")
CHECKPOINT_DIR = Path("checkpoints")
CHECKPOINT_DIR.mkdir(exist_ok=True)

# -- Vocabulary ---------------------------------------------------------------
VOCAB_SIZE  = 10_000   # top-k words + 4 special tokens
MIN_FREQ    = 5        # discard words seen fewer times

# -- Model --------------------------------------------------------------------
D_MODEL     = 512
N_HEAD      = 8
N_LAYERS    = 4        # transformer decoder layers
DIM_FFN     = 2048
DROPOUT     = 0.1
MAX_SEQ_LEN = 42       # <SOS> + up to 50 words + <EOS>

# -- Training -----------------------------------------------------------------
BATCH_SIZE  = 64       # Reduced from 64 for Colab T4 (better memory + throughput)
ACCUMULATION_STEPS = 2  # Effective batch_size = 64 * 2 = 128
NUM_EPOCHS  = 20
LR          = 3e-4
GRAD_CLIP   = 5.0
NUM_WORKERS = 4        # For Colab Linux (use 0 on Windows Jupyter). Set in DataLoader below.

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
AMP    = DEVICE.type == "cuda"   # mixed-precision only on GPU

print(f"Device : {DEVICE}")
if DEVICE.type == "cuda":
    print(f"GPU    : {torch.cuda.get_device_name(0)}")
print(f"AMP    : {AMP}")
print(f"Batch  : {BATCH_SIZE}")


'DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")\nAMP    = DEVICE.type == "cuda"   # mixed-precision only on GPU\n\nprint(f"Device : {DEVICE}")\nif DEVICE.type == "cuda":\n    print(f"GPU    : {torch.cuda.get_device_name(0)}")\nprint(f"AMP    : {AMP}")'

## Vocabulary

In [6]:
class Vocabulary:
    '''Word-level vocabulary with four special tokens: <PAD>, <SOS>, <EOS>, <UNK>.'''
    PAD, SOS, EOS, UNK = "<PAD>", "<SOS>", "<EOS>", "<UNK>"

    def __init__(self):
        self.w2i, self.i2w = {}, {}
        for tok in (self.PAD, self.SOS, self.EOS, self.UNK):
            self._add(tok)

    def _add(self, w):
        if w not in self.w2i:
            idx = len(self.w2i)
            self.w2i[w] = idx
            self.i2w[idx] = w

    @staticmethod
    def tokenize(text: str):
        text = text.lower()
        text = re.sub(r"[^a-z0-9\s]", " ", text)
        return text.split()

    def build(self, captions, max_size=10_000, min_freq=5):
        cnt = Counter()
        for cap in captions:
            cnt.update(self.tokenize(cap))
        for word, freq in cnt.most_common(max_size - 4):
            if freq < min_freq:
                break
            self._add(word)
        print(f"Vocabulary built: {len(self.w2i):,} words")

    def encode(self, text):
        unk = self.w2i[self.UNK]
        return [self.w2i.get(w, unk) for w in self.tokenize(text)]

    def decode(self, ids):
        out = []
        for i in ids:
            w = self.i2w.get(i, self.UNK)
            if w == self.EOS:
                break
            if w not in (self.PAD, self.SOS):
                out.append(w)
        return " ".join(out)

    def __len__(self):
        return len(self.w2i)

    @property
    def pad_idx(self): return self.w2i[self.PAD]
    @property
    def sos_idx(self): return self.w2i[self.SOS]
    @property
    def eos_idx(self): return self.w2i[self.EOS]

In [ ]:
# Load annotation JSON files
'''print("Loading COCO annotations...")
with open(TRAIN_ANN) as f:
    train_data = json.load(f)
with open(VAL_ANN) as f:
    val_data = json.load(f)

# image_id -> file_name maps
train_id2file = {img["id"]: img["file_name"] for img in train_data["images"]}
val_id2file   = {img["id"]: img["file_name"] for img in val_data  ["images"]}

# Create Hugging Face datasets from local COCO data
print("Creating HF datasets from local COCO files...")

# Prepare training data
train_samples = {
    "image_path": [],
    "caption": []
}
for ann in train_data["annotations"]:
    image_id = ann["image_id"]
    if image_id in train_id2file:
        img_path = str(TRAIN_IMG_DIR / train_id2file[image_id])
        train_samples["image_path"].append(img_path)
        train_samples["caption"].append(ann["caption"])

# Prepare validation data
val_samples = {
    "image_path": [],
    "caption": []
}
for ann in val_data["annotations"]:
    image_id = ann["image_id"]
    if image_id in val_id2file:
        img_path = str(VAL_IMG_DIR / val_id2file[image_id])
        val_samples["image_path"].append(img_path)
        val_samples["caption"].append(ann["caption"])

# Create datasets
train_hf_ds = Dataset.from_dict(train_samples)
val_hf_ds = Dataset.from_dict(val_samples)

# Combine into DatasetDict
dataset = DatasetDict({
    "train": train_hf_ds,
    "validation": val_hf_ds
})

print(f"Train samples: {len(train_hf_ds):,} | Val samples: {len(val_hf_ds):,}")

# Push to Hugging Face Hub (one-time operation)
print("\nPushing dataset to Hugging Face Hub...")
# Replace "your-username" with your actual HF username
HF_DATASET_NAME = "vexyiwnl/coco-2017-captions"  # Change this to your HF username!
dataset.push_to_hub(HF_DATASET_NAME, private=False)
print(f"✓ Dataset pushed to: https://huggingface.co/datasets/{HF_DATASET_NAME}")

# Build vocabulary from all training captions
vocab = Vocabulary()
vocab.build(
    train_samples["caption"],
    max_size=VOCAB_SIZE,
    min_freq=MIN_FREQ,
)'''

Loading COCO annotations...
Creating HF datasets from local COCO files...
Train samples: 591,753 | Val samples: 25,014

Pushing dataset to Hugging Face Hub...


Setting num_proc from 1 back to 1 for the train split to disable multiprocessing as it only contains one shard.


Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

...

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

No files have been modified since last commit. Skipping to prevent empty commit.


✓ Dataset pushed to: https://huggingface.co/datasets/vexyiwnl/coco-2017-captions
Vocabulary built: 10,000 words


In [ ]:
'''print("Loading COCO annotations...")
with open(TRAIN_ANN) as f:
    train_data = json.load(f)
with open(VAL_ANN) as f:
    val_data = json.load(f)

# image_id -> file_name maps
train_id2file = {img["id"]: img["file_name"] for img in train_data["images"]}
val_id2file   = {img["id"]: img["file_name"] for img in val_data  ["images"]}

# ===== CREATE AND UPLOAD TO HF HUB (RUN ONCE) =====
# Prepare training data
train_samples = {
    "image_path": [],
    "caption": []
}
for ann in train_data["annotations"]:
    image_id = ann["image_id"]
    if image_id in train_id2file:
        img_path = str(TRAIN_IMG_DIR / train_id2file[image_id])
        train_samples["image_path"].append(img_path)
        train_samples["caption"].append(ann["caption"])

# Prepare validation data
val_samples = {
    "image_path": [],
    "caption": []
}
for ann in val_data["annotations"]:
    image_id = ann["image_id"]
    if image_id in val_id2file:
        img_path = str(VAL_IMG_DIR / val_id2file[image_id])
        val_samples["image_path"].append(img_path)
        val_samples["caption"].append(ann["caption"])

# Create HF datasets
from datasets import Dataset, DatasetDict
train_hf_ds = Dataset.from_dict(train_samples)
val_hf_ds = Dataset.from_dict(val_samples)

dataset = DatasetDict({
    "train": train_hf_ds,
    "validation": val_hf_ds
})

print(f"Train: {len(train_hf_ds):,} | Val: {len(val_hf_ds):,}")

# Push to HF Hub (CHANGE "ankur-username" to your HF username)
print("\nPushing to Hugging Face Hub...")
HF_DATASET_NAME = "vexyiwnl/coco-2017-captions"  # ← EDIT THIS with your username!
try:
    dataset.push_to_hub(HF_DATASET_NAME, private=False)
    print(f"✓ Success! Dataset at: https://huggingface.co/datasets/{HF_DATASET_NAME}")
except Exception as e:
    print(f"Error: {e}")
    print("Make sure you edited HF_DATASET_NAME with your username!")

# Build vocabulary
vocab = Vocabulary()
vocab.build(
    train_samples["caption"],
    max_size=VOCAB_SIZE,
    min_freq=MIN_FREQ,
)'''

Loading COCO annotations...
Train: 591,753 | Val: 25,014

Pushing to Hugging Face Hub...


Setting num_proc from 1 back to 1 for the train split to disable multiprocessing as it only contains one shard.


Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

...

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

No files have been modified since last commit. Skipping to prevent empty commit.


✓ Success! Dataset at: https://huggingface.co/datasets/vexyiwnl/coco-2017-captions
Vocabulary built: 10,000 words


In [ ]:
# ===== FOR FUTURE RUNS: Load from HF Hub (comment out cell above) =====
from datasets import load_dataset
 
HF_DATASET_NAME = "ankur-username/coco-2017-captions"  # ← CHANGE to your username
CACHE_DIR = "/root/.cache/huggingface/datasets"  # Colab persistent cache (MUCH faster after 1st run)
 
print(f"Loading {HF_DATASET_NAME} from Hugging Face Hub...")
dataset = load_dataset(HF_DATASET_NAME, cache_dir=CACHE_DIR)
 
# Extract captions for vocabulary
train_samples = {
    "image_path": dataset["train"]["image_path"],
    "caption": dataset["train"]["caption"]
}

val_samples = {
    "image_path": dataset["validation"]["image_path"],
    "caption": dataset["validation"]["caption"]
}
 
# Build vocabulary
vocab = Vocabulary()
vocab.build(
    train_samples["caption"],
    max_size=VOCAB_SIZE,
    min_freq=MIN_FREQ,
)
 
print(f"✓ Loaded from HF Hub: {len(dataset['train'])} train, {len(dataset['validation'])} val")

In [ ]:
# Convert HF dataset to format compatible with COCOCaptionDataset
print("Converting HF dataset to training format...")

# Create dictionaries for DataLoader compatibility
data_dict_train = {
    "annotations": [
        {"image_id": i, "caption": cap}
        for i, cap in enumerate(train_samples["caption"])
    ],
    "images": [
        {"id": i, "file_name": path}
        for i, path in enumerate(train_samples["image_path"])
    ]
}

data_dict_val = {
    "annotations": [
        {"image_id": i, "caption": cap}
        for i, cap in enumerate(val_samples["caption"])
    ],
    "images": [
        {"id": i, "file_name": path}
        for i, path in enumerate(val_samples["image_path"])
    ]
}

# Create id2file mappings
train_id2file_new = {img["id"]: img["file_name"] for img in data_dict_train["images"]}
val_id2file_new = {img["id"]: img["file_name"] for img in data_dict_val["images"]}

print(f"✓ Converted: {len(train_id2file_new)} train, {len(val_id2file_new)} val image paths")

## Dataset & DataLoaders

In [ ]:
class COCOCaptionDataset(Dataset):
    '''
    One sample = (image tensor, caption tensor).
    All 5 captions per image are exposed as separate training pairs,
    giving ~590K pairs from 118K images.

    Caption tensor layout: [<SOS>, w1, w2, ..., wn, <EOS>]  (truncated to max_len)
    '''
    def __init__(self, data, id2file, img_dir, vocab, transform,
                 max_len=MAX_SEQ_LEN):
        self.vocab     = vocab
        self.transform = transform
        self.img_dir   = Path(img_dir)
        self.max_len   = max_len
        self.samples   = [
            (id2file[a["image_id"]], a["caption"])
            for a in data["annotations"]
        ]

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        fname, cap = self.samples[idx]
        
        # Try to load real image; if not found, generate random PIL image (for testing)
        img_path = self.img_dir / fname
        try:
            img = Image.open(img_path).convert("RGB")
        except (FileNotFoundError, IOError):
            # Generate random PIL image if file doesn't exist (for testing without COCO dataset)
            import numpy as np
            random_array = np.random.randint(0, 256, (224, 224, 3), dtype=np.uint8)
            img = Image.fromarray(random_array, 'RGB')
        
        img = self.transform(img)
        
        ids = ([self.vocab.sos_idx]
               + self.vocab.encode(cap)
               + [self.vocab.eos_idx])
        ids = ids[: self.max_len]
        return img, torch.tensor(ids, dtype=torch.long)

In [ ]:
def collate_fn(batch):
    '''Pad caption tensors within a batch to equal length.'''
    imgs, caps = zip(*batch)
    return (torch.stack(imgs),
            pad_sequence(caps, batch_first=True, padding_value=vocab.pad_idx))


# Image transforms
train_tf = transforms.Compose([
    transforms.RandomResizedCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])
val_tf = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

# Datasets (now using HF dataset converted to compatible format)
train_ds = COCOCaptionDataset(data_dict_train, train_id2file_new, TRAIN_IMG_DIR, vocab, train_tf)
val_ds   = COCOCaptionDataset(data_dict_val,   val_id2file_new,   VAL_IMG_DIR,   vocab, val_tf)

# DataLoaders — use NUM_WORKERS from config (set in configuration cell)
train_loader = DataLoader(train_ds, BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=True, collate_fn=collate_fn)
val_loader   = DataLoader(val_ds,   BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True, collate_fn=collate_fn)

print(f"Train : {len(train_ds):,} samples / {len(train_loader):,} batches @ batch_size={BATCH_SIZE}, num_workers={NUM_WORKERS}")
print(f"Val   : {len(val_ds):,} samples / {len(val_loader):,} batches")


Train : 591,753 samples / 9,247 batches
Val   : 25,014 samples / 391 batches


In [ ]:
# ===== OPTIONAL: Cache HF dataset locally for faster loading =====
# Run this ONCE to download and cache. Then comment out.

import os
import shutil
from datasets import load_dataset

CACHE_DIR = "/root/.cache/huggingface/datasets"  # Colab's persistent cache
HF_DATASET_NAME = "vexyiwnl/coco-2017-captions"  # Your HF dataset

print(f"Caching {HF_DATASET_NAME} to {CACHE_DIR}...")
print("(This is a one-time operation. Subsequent runs will load from cache.)\n")

# Load with caching enabled (HF datasets library caches automatically)
dataset = load_dataset(HF_DATASET_NAME, cache_dir=CACHE_DIR)

print(f"✓ Dataset cached!")
print(f"  Train: {len(dataset['train'])} samples")
print(f"  Val:   {len(dataset['validation'])} samples")
print(f"\nNext time, just run the normal data loading cell — it will use the cache.")


## Model
### 1 — Visual Encoder (ResNet-101)

In [10]:
class ImageEncoder(nn.Module):
    '''
    ResNet-101 visual encoder.

    Pipeline:
        (B, 3, 224, 224)
        ->  ResNet-101 up to layer4  ->  (B, 2048, 7, 7)
        ->  reshape                  ->  (B, 49, 2048)     [49 spatial patch tokens]
        ->  Linear + LayerNorm       ->  (B, 49, d_model)  [cross-attention memory]

    The full backbone is frozen except the last `fine_tune` residual groups,
    which are unfrozen for task-specific fine-tuning.
    '''
    def __init__(self, d_model: int, fine_tune: int = 2):
        super().__init__()
        bb = models.resnet101(weights=models.ResNet101_Weights.DEFAULT)
        # Strip avgpool and fc; keep the convolutional stack
        self.backbone = nn.Sequential(
            bb.conv1, bb.bn1, bb.relu, bb.maxpool,
            bb.layer1, bb.layer2, bb.layer3, bb.layer4,
        )
        # Project 2048-d CNN features to d_model
        self.proj = nn.Sequential(
            nn.Linear(2048, d_model),
            nn.LayerNorm(d_model),
        )
        # Freeze entire backbone, then selectively unfreeze last layers
        for p in self.backbone.parameters():
            p.requires_grad = False
        for layer in list(self.backbone.children())[-fine_tune:]:
            for p in layer.parameters():
                p.requires_grad = True

    def forward(self, x):                                    # (B, 3, 224, 224)
        f = self.backbone(x)                                 # (B, 2048, 7, 7)
        B, C, H, W = f.shape
        f = f.permute(0, 2, 3, 1).reshape(B, H * W, C)      # (B, 49, 2048)
        return self.proj(f)                                  # (B, 49, d_model)

### 2 — Positional Encoding

In [11]:
class PositionalEncoding(nn.Module):
    '''Sinusoidal positional encoding (Vaswani et al., 2017).'''

    def __init__(self, d_model: int, dropout: float = 0.1, max_len: int = 512):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        pe  = torch.zeros(max_len, d_model)
        pos = torch.arange(max_len).unsqueeze(1).float()     # (max_len, 1)
        div = torch.exp(
            torch.arange(0, d_model, 2).float()
            * (-math.log(10_000.0) / d_model)
        )
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer("pe", pe.unsqueeze(1))          # (max_len, 1, d_model)

    def forward(self, x):          # x: (T, B, d_model)
        return self.dropout(x + self.pe[: x.size(0)])

### 3 — Transformer Decoder

In [12]:
class CaptionDecoder(nn.Module):
    '''
    6-layer autoregressive Transformer decoder.

    Each layer (in order):
      1. Masked multi-head self-attention  -- tokens only attend to past positions
      2. Multi-head cross-attention        -- tokens attend to all encoder patch tokens
      3. Position-wise feed-forward network

    Uses Pre-LN (norm_first=True) for more stable gradient flow.
    '''

    def __init__(self, vocab_size, d_model=512, n_head=8, n_layers=6,
                 dim_ffn=2048, dropout=0.1, max_len=512, pad_idx=0):
        super().__init__()
        self.scale   = math.sqrt(d_model)
        self.embed   = nn.Embedding(vocab_size, d_model, padding_idx=pad_idx)
        self.pos_enc = PositionalEncoding(d_model, dropout, max_len)

        dec_layer = nn.TransformerDecoderLayer(
            d_model=d_model, nhead=n_head,
            dim_feedforward=dim_ffn, dropout=dropout,
            batch_first=False,   # (seq, batch, dim) convention
            norm_first=True,     # Pre-LN
        )
        self.transformer = nn.TransformerDecoder(
            dec_layer, num_layers=n_layers, norm=nn.LayerNorm(d_model)
        )
        self.fc_out = nn.Linear(d_model, vocab_size)

        # Weight initialisation
        nn.init.uniform_(self.embed.weight, -0.1, 0.1)
        nn.init.xavier_uniform_(self.fc_out.weight)
        nn.init.zeros_(self.fc_out.bias)

    @staticmethod
    def _causal_mask(sz, device):
        '''Upper-triangular boolean mask: True -> -inf (prevents attending to future tokens).'''
        return torch.triu(torch.ones(sz, sz, device=device), diagonal=1).bool()

    def forward(self, tgt_ids, memory):
        '''
        tgt_ids : (B, T)            teacher-forced input token indices
        memory  : (B, S, d_model)   encoder spatial features
        returns   (B, T, vocab_size) logits
        '''
        T        = tgt_ids.size(1)
        # Embed + scale + add positional encoding; switch to (T, B, d)
        tgt      = self.pos_enc(
            self.embed(tgt_ids).transpose(0, 1) * self.scale
        )
        mem      = memory.transpose(0, 1)                     # (S, B, d)
        tgt_mask = self._causal_mask(T, tgt_ids.device)
        pad_mask = (tgt_ids == self.embed.padding_idx)        # (B, T) bool

        out = self.transformer(
            tgt, mem,
            tgt_mask=tgt_mask,
            tgt_key_padding_mask=pad_mask,
        )                                                      # (T, B, d)
        return self.fc_out(out.transpose(0, 1))               # (B, T, V)

### 4 — Full Model

In [ ]:
class ImageCaptioner(nn.Module):
    '''End-to-end model: ResNet-101 visual encoder + 6-layer Transformer decoder.'''

    def __init__(self, vocab_size, d_model=D_MODEL, n_head=N_HEAD,
                 n_layers=N_LAYERS, dim_ffn=DIM_FFN, dropout=DROPOUT,
                 max_len=MAX_SEQ_LEN + 10, pad_idx=0, fine_tune=1):
        super().__init__()
        self.encoder = ImageEncoder(d_model, fine_tune)
        self.decoder = CaptionDecoder(
            vocab_size, d_model, n_head, n_layers,
            dim_ffn, dropout, max_len, pad_idx
        )

    def forward(self, images, captions):
        '''
        Teacher-forcing forward pass.

        images   : (B, 3, 224, 224)
        captions : (B, T)  full sequence including <SOS> and <EOS>

        Decoder input  -> captions[:, :-1]  (<SOS> w1 ... w_{n-1})
        Decoder target -> captions[:,  1:]  (w1 ... w_{n-1} <EOS>)
        '''
        memory = self.encoder(images)                          # (B, 49, d_model)
        logits = self.decoder(captions[:, :-1], memory)       # (B, T-1, vocab_size)
        return logits

    # -- Inference ------------------------------------------------------------

    @torch.no_grad()
    def generate(self, image, vocab, max_len=50, beam_size=1):
        '''Generate a caption for a single image tensor (greedy or beam search).'''
        self.eval()
        device = next(self.parameters()).device
        if image.dim() == 3:
            image = image.unsqueeze(0)
        memory = self.encoder(image.to(device))                # (1, 49, d_model)
        if beam_size == 1:
            return self._greedy(memory, vocab, max_len)
        return self._beam(memory, vocab, max_len, beam_size)

    def _greedy(self, memory, vocab, max_len):
        device = memory.device
        tokens = [vocab.sos_idx]
        for _ in range(max_len):
            tgt    = torch.tensor([tokens], device=device)
            logits = self.decoder(tgt, memory)                 # (1, T, V)
            nxt    = logits[0, -1].argmax(-1).item()
            if nxt == vocab.eos_idx:
                break
            tokens.append(nxt)
        return vocab.decode(tokens[1:])                        # strip <SOS>

    def _beam(self, memory, vocab, max_len, k):
        device = memory.device
        beams  = [(0.0, [vocab.sos_idx])]
        done   = []
        for _ in range(max_len):
            cands = []
            for score, seq in beams:
                if seq[-1] == vocab.eos_idx:
                    done.append((score, seq))
                    continue
                tgt  = torch.tensor([seq], device=device)
                lp   = F.log_softmax(
                    self.decoder(tgt, memory)[0, -1], dim=-1
                )
                vals, ids = lp.topk(k)
                for v, i in zip(vals.tolist(), ids.tolist()):
                    cands.append((score + v, seq + [i]))
            if not cands:
                break
            beams = sorted(cands, key=lambda x: x[0], reverse=True)[:k]
        done += beams
        best = max(done, key=lambda x: x[0] / max(len(x[1]), 1))
        return vocab.decode(best[1][1:])                       # strip <SOS>

## Training Utilities

In [14]:
def save_ckpt(model, optimizer, epoch, loss, path):
    torch.save({
        "epoch":     epoch,
        "model":     model.state_dict(),
        "optimizer": optimizer.state_dict(),
        "loss":      loss,
    }, path)
    print(f"  Checkpoint saved -> {path}  (val_loss={loss:.4f})")


def load_ckpt(path, model, optimizer=None):
    ckpt = torch.load(path, map_location=DEVICE)
    model.load_state_dict(ckpt["model"])
    if optimizer:
        optimizer.load_state_dict(ckpt["optimizer"])
    print(f"  Loaded checkpoint -- epoch {ckpt['epoch']}, val_loss={ckpt['loss']:.4f}")
    return ckpt["epoch"], ckpt["loss"]

In [ ]:
def train_one_epoch(model, loader, optimizer, criterion, scaler, scheduler, epoch):
    '''One full training epoch with teacher forcing and AMP.'''
    model.train()
    total_loss, total_toks = 0.0, 0
    bar = tqdm(loader, desc=f"Ep {epoch:02d} [train]", leave=False)

    for i, (imgs, caps) in enumerate(bar):
        imgs, caps = imgs.to(DEVICE), caps.to(DEVICE)

        with torch.cuda.amp.autocast(enabled=AMP):
            logits  = model(imgs, caps)
            targets = caps[:, 1:].contiguous()
            B, T, V = logits.shape
            loss    = criterion(
                logits.reshape(B * T, V),
                targets.reshape(B * T)
            ) / ACCUMULATION_STEPS  # ← Divide loss

        scaler.scale(loss).backward()

        n = (targets != vocab.pad_idx).sum().item()
        total_loss += loss.item() * n * ACCUMULATION_STEPS  # ← Restore for logging
        total_toks += n

        # Update only every ACCUMULATION_STEPS batches
        if (i + 1) % ACCUMULATION_STEPS == 0:
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()
            if scheduler:
                scheduler.step()

        bar.set_postfix(loss=f"{loss.item():.4f}")

    return total_loss / total_toks

In [16]:
@torch.no_grad()
def val_one_epoch(model, loader, criterion, epoch):
    '''One full validation epoch (no gradient updates).'''
    model.eval()
    total_loss, total_toks = 0.0, 0
    bar = tqdm(loader, desc=f"Ep {epoch:02d} [val]  ", leave=False)

    for imgs, caps in bar:
        imgs, caps = imgs.to(DEVICE), caps.to(DEVICE)
        with torch.cuda.amp.autocast(enabled=AMP):
            logits  = model(imgs, caps)
            targets = caps[:, 1:].contiguous()
            B, T, V = logits.shape
            loss    = criterion(
                logits.reshape(B * T, V),
                targets.reshape(B * T)
            )
        n = (targets != vocab.pad_idx).sum().item()
        total_loss += loss.item() * n
        total_toks += n
        bar.set_postfix(loss=f"{loss.item():.4f}")

    return total_loss / total_toks

## Build Model + Optimiser

In [17]:
model = ImageCaptioner(
    vocab_size = len(vocab),
    pad_idx    = vocab.pad_idx,
).to(DEVICE)

# Separate learning-rate groups: encoder backbone (pre-trained) gets 10x smaller LR
optimizer = torch.optim.AdamW([
    {"params": model.encoder.backbone.parameters(), "lr": LR * 0.1},
    {"params": model.encoder.proj.parameters(),     "lr": LR},
    {"params": model.decoder.parameters(),          "lr": LR},
], weight_decay=1e-4)

total_steps = NUM_EPOCHS * len(train_loader)
scheduler   = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=total_steps, eta_min=1e-6
)
scaler    = torch.cuda.amp.GradScaler(enabled=AMP)
criterion = nn.CrossEntropyLoss(ignore_index=vocab.pad_idx, label_smoothing=0.1)

total_p = sum(p.numel() for p in model.parameters())
train_p = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Parameters: {total_p:,} total / {train_p:,} trainable")

Parameters: 79,025,488 total / 77,580,560 trainable


C:\Users\ankur\AppData\Local\Temp\ipykernel_10888\1456533093.py:17: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler    = torch.cuda.amp.GradScaler(enabled=AMP)


In [ ]:
import time

# Quick data loading profiler (run this to diagnose bottleneck)
print("Profiling data loading + forward pass speed...")
start = time.time()
batch_count = 0
data_time = 0
compute_time = 0

for imgs, caps in train_loader:
    batch_start = time.time()
    
    # Time data loading
    data_load_time = batch_start - start
    
    # Time forward pass
    imgs = imgs.to(DEVICE)
    compute_start = time.time()
    with torch.cuda.amp.autocast(enabled=AMP):
        _ = model.encoder(imgs)
    compute_batch_time = time.time() - compute_start
    
    data_time += data_load_time
    compute_time += compute_batch_time
    batch_count += 1
    
    if batch_count >= 10:  # Profile first 10 batches
        break
    start = time.time()

avg_data_time = data_time / batch_count
avg_compute_time = compute_time / batch_count
print(f"\nFirst 10 batches:")
print(f"  Avg data load time:   {avg_data_time*1000:.1f}ms")
print(f"  Avg encoder time:     {avg_compute_time*1000:.1f}ms")
print(f"  Total per batch:      {(avg_data_time + avg_compute_time)*1000:.1f}ms")
print(f"\nDiagnosis:")
if avg_data_time > 500:
    print(f"  ⚠️  Data loading is the bottleneck (>500ms). Dataset may still be streaming from HF Hub.")
    print(f"     Did you cache the dataset? Run the caching cell first.")
else:
    print(f"  ✓ Data loading is fast ({avg_data_time*1000:.0f}ms). GPU compute is the bottleneck.")


In [ ]:
import subprocess
import time

print("Monitoring GPU memory (RTX 4050). Press Ctrl+C to stop.\n")
print("Batch Size: 64 | Accumulation Steps: 2 | Effective Batch: 128")
print("-" * 60)

try:
    while True:
        result = subprocess.run(
            ["nvidia-smi", "--query-gpu=memory.used,memory.total", 
             "--format=csv,noheader,nounits"],
            capture_output=True, text=True
        )
        used, total = map(int, result.stdout.strip().split(','))
        pct = (used / total) * 100
        
        print(f"VRAM: {used:5d}/{total:5d} MB ({pct:.1f}%) ", end="")
        if pct > 90:
            print("⚠️  CRITICAL - Reduce batch size!")
        elif pct > 80:
            print("⚠️  HIGH - Monitor closely")
        else:
            print("✓ Safe")
        
        time.sleep(2)
except KeyboardInterrupt:
    print("\nMonitoring stopped.")

## Training Loop

In [ ]:
history  = {"train": [], "val": []}
best_val = float("inf")

for epoch in range(1, NUM_EPOCHS + 1):
    tr = train_one_epoch(model, train_loader, optimizer, criterion,
                         scaler, scheduler, epoch)
    
    # Validate every 2 epochs instead of every epoch (Part B)
    if epoch % 2 == 0 or epoch == NUM_EPOCHS:
        vl = val_one_epoch(model, val_loader, criterion, epoch)
        history["val"].append(vl)
        
        print(f"Epoch {epoch:02d}/{NUM_EPOCHS}  train={tr:.4f}  val={vl:.4f}")
        
        if vl < best_val:
            best_val = vl
            save_ckpt(model, optimizer, epoch, vl,
                      CHECKPOINT_DIR / "best.pth")
    else:
        print(f"Epoch {epoch:02d}/{NUM_EPOCHS}  train={tr:.4f}  (val skipped)")
    
    history["train"].append(tr)

Ep 01 [train]:   0%|          | 0/9247 [00:00<?, ?it/s]

C:\Users\ankur\AppData\Local\Temp\ipykernel_10888\2350320619.py:10: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=AMP):


Ep 01 [val]  :   0%|          | 0/391 [00:00<?, ?it/s]

Epoch 01/20  train=3.5366  val=3.2728
  Checkpoint saved -> checkpoints\best.pth  (val_loss=3.2728)


Ep 02 [train]:   0%|          | 0/9247 [00:00<?, ?it/s]

Ep 02 [val]  :   0%|          | 0/391 [00:00<?, ?it/s]

Epoch 02/20  train=3.2497  val=3.1795
  Checkpoint saved -> checkpoints\best.pth  (val_loss=3.1795)


Ep 03 [train]:   0%|          | 0/9247 [00:00<?, ?it/s]

In [ ]:
# Loss curve
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(history["train"], label="train")
ax.plot(history["val"],   label="val")
ax.set(xlabel="Epoch", ylabel="CE Loss", title="Training curve")
ax.legend()
fig.tight_layout()
fig.savefig(CHECKPOINT_DIR / "loss_curve.png", dpi=150)
plt.show()

## Inference & Visualisation

In [ ]:
# Load best checkpoint before running inference
load_ckpt(CHECKPOINT_DIR / "best.pth", model)


def caption_image(img_path, beam_size=3):
    '''Open an image file, generate a caption, and display both.'''
    raw    = Image.open(img_path).convert("RGB")
    tensor = val_tf(raw)
    pred   = model.generate(tensor, vocab, max_len=50, beam_size=beam_size)

    fig, ax = plt.subplots(figsize=(6, 5))
    ax.imshow(raw)
    ax.axis("off")
    ax.set_title(pred, fontsize=12, wrap=True)
    plt.tight_layout()
    plt.show()
    return pred


# Sample 5 random validation images
random.seed(42)
sample_paths = random.sample(sorted(VAL_IMG_DIR.iterdir()), 5)
for p in sample_paths:
    cap = caption_image(p, beam_size=3)
    print(cap)